In [1]:
import tensorflow as tf
import numpy as np
import re
import urllib.request

print("TensorFlow version:", tf.__version__)

# Step 2: Download Shakespeare's text dataset
url = "https://www.gutenberg.org/cache/epub/100/pg100.txt"

with urllib.request.urlopen(url) as response:
    text = response.read().decode("utf-8")

print("Dataset downloaded successfully!")
print("Total characters:", len(text))
print("\nFirst 500 characters:\n")
print(text[:500])


TensorFlow version: 2.21.0
Dataset downloaded successfully!
Total characters: 5575099

First 500 characters:

The Project Gutenberg eBook of The Complete Works of William Shakespeare
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are locat


In [2]:

# Step 3: Clean and preprocess the dataset

# Remove Gutenberg's header and footer when their markers exist
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"

start = text.find(start_marker)
if start != -1:
    newline = text.find("\n", start)
    text = text[newline + 1:]

end = text.find(end_marker)
if end != -1:
    text = text[:end]

# Convert all text to lowercase
text = text.lower()

# Remove punctuation and digits, keeping letters and whitespace
text = re.sub(r"[^a-z\s]", "", text)

# Replace repeated whitespace and trim the text
text = re.sub(r"\s+", " ", text).strip()

# Split the cleaned text into individual words
words = text.split()

print("Preprocessing completed!")
print("Total words:", len(words))
print("First 30 words:")
print(words[:30])

# Create a vocabulary of unique words
vocabulary = sorted(set(words))

print("\nUnique words:", len(vocabulary))


Preprocessing completed!
Total words: 962814
First 30 words:
['the', 'complete', 'works', 'of', 'william', 'shakespeare', 'by', 'william', 'shakespeare', 'contents', 'the', 'sonnets', 'alls', 'well', 'that', 'ends', 'well', 'the', 'tragedy', 'of', 'antony', 'and', 'cleopatra', 'as', 'you', 'like', 'it', 'the', 'comedy', 'of']

Unique words: 29874


In [3]:

# Step 4: Convert words into numerical IDs

from tensorflow.keras.preprocessing.text import Tokenizer

# Assign a unique number to each word
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

# Convert the complete text into a list of numbers
token_ids = tokenizer.texts_to_sequences([text])[0]

# Get the total number of unique words
vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)
print("First 20 word IDs:", token_ids[:20])

# Create sequences of five input words and one target word
sequence_length = 5
sequences = []

for i in range(sequence_length, len(token_ids)):
    input_sequence = token_ids[i-sequence_length:i]
    target_word = token_ids[i]
    sequences.append(input_sequence + [target_word])

print("Total training sequences:", len(sequences))
print("Example sequence:", sequences[0])


Vocabulary size: 29875
First 20 word IDs: [1, 3601, 2644, 5, 1210, 9256, 33, 1210, 9256, 1470, 1, 7458, 2036, 54, 10, 1741, 54, 1, 2867, 5]
Total training sequences: 962809
Example sequence: [1, 3601, 2644, 5, 1210, 9256]


In [4]:

# Step 5A: Prepare input (X) and target (y)

import numpy as np
from sklearn.model_selection import train_test_split

sequences = np.array(sequences, dtype=np.int32)

# First five words are inputs; sixth word is the target
X = sequences[:, :-1]
y = sequences[:, -1]

# Split into training and validation sets
# Shuffle=False keeps the earlier text in training and later text in validation.
split_index = int(len(X) * 0.9)

X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

print("Training inputs:", X_train.shape)
print("Training targets:", y_train.shape)
print("Validation inputs:", X_val.shape)
print("Validation targets:", y_val.shape)


Training inputs: (866528, 5)
Training targets: (866528,)
Validation inputs: (96281, 5)
Validation targets: (96281,)


In [5]:

# Step 5B: Build the LSTM model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64),
    LSTM(128),
    Dropout(0.2),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:

# Build the model using sequences of five words
model.build(input_shape=(None, sequence_length))

# Display the actual architecture and parameter count
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 64)          │     1,912,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 29875)          │     3,853,875 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,864,691 (22.37 MB)

 Trainable params: 5,864,691 (22.37 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:

# Step 6: Train the LSTM model

from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# Use a manageable subset for the first experiment
X_train_small = X_train[:150000]
y_train_small = y_train[:150000]

X_val_small = X_val[:20000]
y_val_small = y_val[:20000]

# Stop training when validation loss stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

# Save the model with the best validation loss
checkpoint = ModelCheckpoint(
    "best_lstm_text_model.keras",
    monitor="val_loss",
    save_best_only=True
)

# Train the model
history = model.fit(
    X_train_small,
    y_train_small,
    validation_data=(X_val_small, y_val_small),
    epochs=5,
    batch_size=256,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

print("Training completed!")


Epoch 1/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 125s 212ms/step - accuracy: 0.0304 - loss: 7.0753 - val_accuracy: 0.0291 - val_loss: 7.1582
Epoch 2/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 139s 206ms/step - accuracy: 0.0431 - loss: 6.5913 - val_accuracy: 0.0397 - val_loss: 7.1140
Epoch 3/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 138s 199ms/step - accuracy: 0.0536 - loss: 6.4172 - val_accuracy: 0.0487 - val_loss: 7.0513
Epoch 4/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 117s 200ms/step - accuracy: 0.0676 - loss: 6.2343 - val_accuracy: 0.0645 - val_loss: 6.9939
Epoch 5/5
586/586 ━━━━━━━━━━━━━━━━━━━━ 143s 201ms/step - accuracy: 0.0809 - loss: 6.0638 - val_accuracy: 0.0734 - val_loss: 7.0008
Training completed!


In [8]:

# Step 7: Generate text using the trained LSTM model

def generate_text(seed_text, next_words=30, temperature=0.8):
    result = seed_text.lower().split()

    for _ in range(next_words):
        # Keep the most recent five words
        context = result[-sequence_length:]

        # Convert words into their numerical IDs
        token_ids = [
            tokenizer.word_index.get(word, 0)
            for word in context
        ]

        # Pad the input if fewer than five words are available
        token_ids = [0] * (sequence_length - len(token_ids)) + token_ids

        # Predict probabilities for the next word
        predictions = model.predict(
            np.array([token_ids]), verbose=0
        )[0]

        # Apply temperature to control randomness
        predictions = np.log(predictions + 1e-8) / temperature
        probabilities = tf.nn.softmax(predictions).numpy()

        # Sample the next word using predicted probabilities
        next_id = np.random.choice(len(probabilities), p=probabilities)

        # Convert the predicted ID back to a word
        next_word = tokenizer.index_word.get(next_id)

        if next_word is None:
            continue

        result.append(next_word)

    return " ".join(result)


# Generate text from three different seed inputs
seeds = [
    "the king is",
    "to be or not to",
    "my lord and"
]

for seed in seeds:
    print("\nSeed:", seed)
    print("Generated:", generate_text(seed, next_words=30))



Seed: the king is
Generated: the king is my great noble hand let thee is the court of our eyes and make such no part of the ho shall are none of as more fame i will faith

Seed: to be or not to
Generated: to be or not to his very veins out of love to these girl which more wholesome with it and embrace her marry will thou best a roman more exit in this soldiers let your

Seed: my lord and
Generated: my lord and care each in gods upon a very greets to thy people here should much begins not within joy to thee this fear to your man is him in this was


In [9]:

# Step 9: Save generated text outputs to a file

sample_outputs = """
LSTM Text Generation - Sample Outputs

Seed: the king is
Generated: the king is my great noble hand let thee is the court of our eyes and make such no part of the ho shall are none of as more fame i will faith

Seed: to be or not to
Generated: to be or not to his very veins out of love to these girl which more wholesome with it and embrace her marry will thou best a roman more exit in this soldiers let your

Seed: my lord and
Generated: my lord and care each in gods upon a very greets to thy people here should much begins not within joy to thee this fear to your man is him in this was
"""

with open("sample_output.txt", "w", encoding="utf-8") as file:
    file.write(sample_outputs)

print("Sample outputs saved successfully!")


Sample outputs saved successfully!


In [10]:

from google.colab import files

files.download("sample_output.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>